In [ ]:
# Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved.
#
# SPDX-License-Identifier: BSD-3-Clause
#
# Portions of this file consist of AI-generated content. AI-assisted
# content has been reviewed and validated by the authors.

# AMD Advancing AI — Physical AI Agent Workshop
### Natural language → LLM plan → HIP/ROCm perception → Genesis Franka motion

**Stack:** AMD Strix Halo (Radeon 8060S iGPU) · Llama-3.2-3B (llama.cpp/Vulkan) · custom **HIP/ROCm PyTorch kernels** · Genesis physics.

This notebook is a guided tour of the *exact* agent in `demo_v3_rocm.py`. The perception kernels are pulled **out into editable cells below** — you write them, and the live agent runs *your* version.

By the end you will have concretely answered three questions:

| # | Question | Answered in |
|---|----------|-------------|
| 1 | **Is the LLM actually running?** | § 2 — you send a sentence, the model returns a JSON plan (with inference time) |
| 2 | **Where is the Genesis Franka arm?** | § 3 — a rendered frame of the real gold arm, inline |
| 3 | **What do the HIP kernels output — and can I edit them?** | § 4 — you author the 8 kernels, patch them into the agent, and see their maps inline |

Then § 6 runs the whole loop on a prerecorded command list (including **stack blue on red on green**) and embeds the mission-control video right here. § 7 lets you type your own commands live and saves that run too.

## 1 · The design spine

> **The LLM decides _what_. ROCm kernels compute _where_ and _how_.**

The language model turns *“stack blue on red on green”* into a structured plan. But the numbers that actually move the arm — the grasp point, the cube's true center, whether the fingers have really closed on it — come from **hand-written HIP/ROCm kernels** running on the AMD iGPU:

* `hip_normalize_depth` — raw depth → metric grasp height
* `hip_colorize_normal` — surface normals → approach orientation
* `hip_colorize_seg` — instance IDs → which pixels are which cube
* `hip_color_lock` — chromaticity match → the cube's pixel center, back-projected to the table
* `hip_tactile_reduce` — 8×8 taxel displacement → grip force + “secure” gate
* `hip_rgb_to_gray` — RGB → luminance grayscale (classic GPU reduction)
* `hip_gaussian_blur` — separable Gaussian blur (conv2d)
* `hip_sobel_edges` — Sobel gradient magnitude (edge map)

In § 4 these eight kernels are yours to edit. The 2-panel HUD in § 6 (live scene on the left, an “AI BRAIN” telemetry/plan panel on the right) is just a window onto them doing their job every frame.

### Set up — import the proven module and confirm ROCm is live

In [ ]:
import os, sys, time, json, queue, threading
import numpy as np
import matplotlib.pyplot as plt

# The whole agent lives in this module (same folder). We reuse it verbatim,
# then in Section 4 we override its kernels with the ones you write.
import demo_v3_rocm as demo

print("module     :", demo.__file__)
try:
    import torch

    print("torch      :", torch.__version__)
    print("torch.hip  :", getattr(torch.version, "hip", None))
    print(
        "cuda avail :", torch.cuda.is_available(), "  (True == ROCm HIP device is live)"
    )
    if torch.cuda.is_available():
        print("device     :", torch.cuda.get_device_name(0))
except Exception as e:
    print("torch not available:", e)

print("HIP_DEVICE :", demo.HIP_DEVICE, "   <- kernels dispatch here")

## 2 · Is the LLM actually running? (and when does it kick in?)

`start_llm_server()` launches (or reattaches to) **llama-server** with `-ngl 99` so all layers of Llama-3.2-3B sit on the AMD iGPU via Vulkan. We then send plain-English commands through `parse_plan()`, **timing each inference** and printing the GPU busy % around it — so you can literally see the moment the LLM kicks in and lands on the iGPU.

In [ ]:
demo.start_llm_server()  # idempotent: reattaches if already up (prints 'already running')


def plan_with_telemetry(cmd):
    """Run one planning call, showing the LLM-kick moment + GPU stats."""
    g0 = float(demo.GPU_STATE.get("gpu_pct", 0.0))
    print(f"\nYOU  > {cmd}")
    print("       LLM PLANNING on AMD iGPU ...", end="", flush=True)
    t0 = time.time()
    parsed = demo.parse_plan(cmd)
    ms = (time.time() - t0) * 1000.0
    g1 = float(demo.GPU_STATE.get("gpu_pct", 0.0))
    vram = float(demo.GPU_STATE.get("vram_used_mb", 0.0))
    print(
        f"  done in {ms:8.1f} ms   | GPU {g0:.0f}%% -> {g1:.0f}%%  VRAM {vram:.0f} MB".replace(
            "%%", "%"
        )
    )
    print("       plan >", json.dumps(parsed.get("plan", [])))
    return parsed


for cmd in [
    "stack blue on red on green",
    "sort the cubes",
    "pick the blue cube and put it on the left",
]:
    plan_with_telemetry(cmd)

## 3 · Where is the Genesis Franka arm?

`setup_scene()` builds the Genesis physics world on the `amdgpu` backend: a table plane, three colored cubes, the **Franka Emika Panda** arm (recolored flat **gold** via a surface override — same look as the reference demo), a camera, and the 8×8 ElastomerTaxel tactile pads on each fingertip. We build it **once** here and reuse it for the rest of the notebook, then render one camera frame so you can see the arm.

We also start the GPU-telemetry worker thread now, exactly as the live demo does, so the HUD panel has real AMD iGPU numbers to show in § 6.

In [ ]:
# --- GPU telemetry worker thread (same as demo_v3_rocm.main) ------------
gpu_stop = threading.Event()
threading.Thread(target=demo.gpu_monitor, args=(gpu_stop,), daemon=True).start()

# --- build the Genesis scene ONCE ---------------------------------------
if "scene" not in globals():
    scene, franka, cam, motors_dof, fingers_dof = demo.setup_scene()
    demo.go_home(franka, scene, cam, queue.Queue(maxsize=4), motors_dof, fingers_dof)
    print("[SCENE] built + arm homed  (arm colour =", demo.ARM_COLOR_RGB, ")")
else:
    print("[SCENE] already built — reusing existing scene")

# --- render one frame of the real scene ---------------------------------
rgb, *_ = cam.render(rgb=True, depth=False, segmentation=False, normal=False)
rgb = rgb.cpu().numpy() if hasattr(rgb, "cpu") else np.asarray(rgb)
plt.figure(figsize=(8, 6))
plt.imshow(rgb)
plt.axis("off")
plt.title("Genesis — gold Franka Panda + red/green/blue cubes (live camera render)")
plt.show()

## 4 · The HIP/ROCm kernels — **yours to write**

This is the teaching core. The eight perception + tactile kernels are defined right here in the notebook, not hidden in a `.py`. **Edit the math, re-run the cell, and the bottom of the cell monkeypatches your versions straight into the running agent** (`demo.hip_* = your_function`). Because the agent calls these kernels by name, § 6 will then move the arm using *your* code.

Each kernel takes plain NumPy from the Genesis camera/tactile pads, does the heavy lifting on the `HIP_DEVICE` (the ROCm iGPU), and returns something the agent consumes:

* **K1 `hip_normalize_depth`** → 8-bit metric depth band
* **K2 `hip_colorize_normal`** → surface-normal RGB (approach orientation)
* **K3 `hip_colorize_seg`** → per-instance color hash (which cube is which)
* **K3b `hip_color_lock`** → chromaticity match → pixel bbox/centroid of a target cube
* **K4 `hip_tactile_reduce`** → taxel field → grip force + `secure` bool that gates the grasp
* **K5 `hip_rgb_to_gray`** → BT.601 luminance (the classic RGB→gray reduction)
* **K6 `hip_gaussian_blur`** → separable Gaussian conv2d (denoise before edges)
* **K7 `hip_sobel_edges`** → Sobel gradient magnitude (edge map)

> Try it: widen `tol` in `hip_color_lock`, or drop `secure_taxels` in `hip_tactile_reduce`, re-run this cell, then re-run § 6 and watch the behavior change.

In [ ]:
# ============================================================================
#  EDIT ME — these eight kernels drive the live agent after you run this cell.
# ============================================================================
import numpy as np, torch

# Reuse the tuned constants + device from the module so your defaults match the
# calibrated agent (you can hard-code different numbers if you want).
from demo_v3_rocm import (
    HIP_DEVICE,
    GRIP_STIFFNESS_N_PER_M,
    COLOR_LOCK_TOL,
    CONTACT_THRESH_M,
    CONTACT_SECURE_TAXELS,
)


def hip_normalize_depth(depth_np, near=0.1, far=5.0):
    """K1: raw depth buffer -> uint8 metric band on the iGPU."""
    d = torch.from_numpy(np.asarray(depth_np, dtype=np.float32)).to(HIP_DEVICE)
    if d.ndim == 3:
        d = d[:, :, 0]
    return ((d - near) / (far - near)).clamp(0, 1).mul(255).byte().cpu().numpy()


def hip_colorize_normal(normal_np):
    """K2: surface normals -> RGB (encodes approach orientation)."""
    n = torch.from_numpy(np.asarray(normal_np, dtype=np.float32)).to(HIP_DEVICE)
    if n.ndim == 4:
        n = n[0]
    out = (n + 1.0) / 2.0 if n.min().item() < -0.05 else n
    mn, mx = out.min(), out.max()
    if (mx - mn).item() > 1e-4:
        out = (out - mn) / (mx - mn)
    return (out * 255).clamp(0, 255).byte().cpu().numpy()


def hip_colorize_seg(seg_np):
    """K3: instance-id buffer -> deterministic RGB hash per object."""
    arr = np.asarray(seg_np)
    if arr.ndim == 4:
        arr = arr[0]
    ids = arr[..., 0].astype(np.int32) if arr.ndim == 3 else arr.astype(np.int32)
    s = torch.from_numpy(ids).to(HIP_DEVICE)
    r = ((s * 37 + 11) % 256).byte()
    g = ((s * 79 + 43) % 256).byte()
    b = ((s * 131 + 97) % 256).byte()
    return torch.stack([r, g, b], dim=-1).cpu().numpy()


def hip_color_lock(rgb_uint8, target, tol=COLOR_LOCK_TOL):
    """K3b: find a target-colored cube in image space -> {n, bbox, cx, cy}."""
    a = np.asarray(rgb_uint8)
    if a.ndim != 3 or a.shape[-1] < 3:
        return {"n": 0, "bbox": None, "cx": None, "cy": None}
    t = torch.as_tensor(a[..., :3].astype(np.float32)).to(HIP_DEVICE) / 255.0
    tgt = torch.tensor(target, dtype=torch.float32, device=t.device)
    mask = torch.linalg.norm(t - tgt, dim=-1) < tol
    n = int(mask.sum().item())
    if n < 8:
        return {"n": n, "bbox": None, "cx": None, "cy": None}
    ys, xs = torch.where(mask)
    return {
        "n": n,
        "bbox": (int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())),
        "cx": int(xs.float().mean().item()),
        "cy": int(ys.float().mean().item()),
    }


def hip_tactile_reduce(
    left_disp,
    right_disp,
    contact_thresh_m=CONTACT_THRESH_M,
    secure_taxels=CONTACT_SECURE_TAXELS,
):
    """K4: 8x8 taxel displacement fields -> grip force + secure gate."""
    l = (
        torch.as_tensor(np.asarray(left_disp, dtype=np.float32))
        .to(HIP_DEVICE)
        .reshape(-1, 3)
    )
    r = (
        torch.as_tensor(np.asarray(right_disp, dtype=np.float32))
        .to(HIP_DEVICE)
        .reshape(-1, 3)
    )
    lmag = torch.linalg.norm(l, dim=-1)
    rmag = torch.linalg.norm(r, dim=-1)
    n_contact = int(
        (lmag > contact_thresh_m).sum().item() + (rmag > contact_thresh_m).sum().item()
    )
    grip_force_N = float((lmag.sum() + rmag.sum()).item() * GRIP_STIFFNESS_N_PER_M)
    peak_mm = float(max(lmag.max().item(), rmag.max().item()) * 1000.0)
    return {
        "n_contact": n_contact,
        "n_taxels": int(lmag.numel() + rmag.numel()),
        "grip_force_N": grip_force_N,
        "peak_mm": peak_mm,
        "secure": n_contact >= secure_taxels,
    }


def hip_rgb_to_gray(rgb_uint8):
    """K5: RGB -> BT.601 luminance grayscale on the iGPU. Returns HxW uint8."""
    a = np.asarray(rgb_uint8)
    if a.ndim == 4:
        a = a[0]
    t = torch.as_tensor(a[..., :3].astype(np.float32)).to(HIP_DEVICE)
    w = torch.tensor([0.299, 0.587, 0.114], dtype=torch.float32, device=t.device)
    return (t * w).sum(-1).clamp(0, 255).byte().cpu().numpy()


def hip_gaussian_blur(gray_uint8, ksize=5, sigma=1.4):
    """K6: separable Gaussian blur via two depthwise conv2d passes."""
    a = np.asarray(gray_uint8, dtype=np.float32)
    if a.ndim == 3:
        a = a[..., 0]
    t = torch.as_tensor(a).to(HIP_DEVICE)[None, None]
    ax = torch.arange(ksize, device=t.device, dtype=torch.float32) - (ksize - 1) / 2.0
    k1 = torch.exp(-(ax * ax) / (2.0 * sigma * sigma))
    k1 = k1 / k1.sum()
    pad = ksize // 2
    t = torch.nn.functional.conv2d(t, k1.view(1, 1, 1, ksize), padding=(0, pad))
    t = torch.nn.functional.conv2d(t, k1.view(1, 1, ksize, 1), padding=(pad, 0))
    return t[0, 0].clamp(0, 255).byte().cpu().numpy()


def hip_sobel_edges(gray_uint8):
    """K7: Sobel gradient magnitude (two conv2d + hypot), normalised."""
    a = np.asarray(gray_uint8, dtype=np.float32)
    if a.ndim == 3:
        a = a[..., 0]
    t = torch.as_tensor(a).to(HIP_DEVICE)[None, None]
    gx = torch.tensor(
        [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32, device=t.device
    ).view(1, 1, 3, 3)
    gy = torch.tensor(
        [[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32, device=t.device
    ).view(1, 1, 3, 3)
    ex = torch.nn.functional.conv2d(t, gx, padding=1)
    ey = torch.nn.functional.conv2d(t, gy, padding=1)
    mag = torch.sqrt(ex * ex + ey * ey)
    mag = mag / (mag.max() + 1e-6) * 255.0
    return mag[0, 0].clamp(0, 255).byte().cpu().numpy()


# --- wire YOUR kernels into the running agent -------------------------------
_KERNELS = [
    "hip_normalize_depth",
    "hip_colorize_normal",
    "hip_colorize_seg",
    "hip_color_lock",
    "hip_tactile_reduce",
    "hip_rgb_to_gray",
    "hip_gaussian_blur",
    "hip_sobel_edges",
]
for _n in _KERNELS:
    setattr(demo, _n, globals()[_n])
print("patched into the live agent:", ", ".join(_KERNELS))
print("Section 6 will now move the arm using the kernels defined in THIS cell.")

### See your kernels' output

Grab all four camera modalities (RGB, depth, segmentation, surface normals) in one render and push them through the kernels you just wrote. Each panel is the raw output of one HIP kernel on the iGPU — the exact perception the arm relies on.

In [ ]:
# _grab_modalities does one render; demo.hip_* now point at YOUR functions.
rgb_np, depth_np, seg_np, nrm_np = demo._grab_modalities(cam)

depth_u8 = demo.hip_normalize_depth(depth_np)  # K1
normal_u8 = demo.hip_colorize_normal(nrm_np)  # K2
seg_u8 = demo.hip_colorize_seg(seg_np)  # K3
lock = demo.hip_color_lock(
    np.asarray(rgb_np, np.uint8)[..., :3], demo.TARGET_COLOR_RGB["red"]
)  # K3b: find the red cube
gray_u8 = demo.hip_rgb_to_gray(rgb_np)  # K5
blur_u8 = demo.hip_gaussian_blur(gray_u8)  # K6  (blur the gray)
sobel_u8 = demo.hip_sobel_edges(blur_u8)  # K7  (edges on the blur)

print("K3b color-lock (red):", lock)

fig, ax = plt.subplots(2, 4, figsize=(16, 8))
fig.patch.set_facecolor("#12121a")
panels = [
    (rgb_np, "RGB  (camera)"),
    (depth_u8, "K1 · hip_normalize_depth"),
    (normal_u8, "K2 · hip_colorize_normal"),
    (seg_u8, "K3 · hip_colorize_seg"),
    (gray_u8, "K5 · hip_rgb_to_gray"),
    (blur_u8, "K6 · hip_gaussian_blur"),
    (sobel_u8, "K7 · hip_sobel_edges"),
    (None, ""),
]
cmaps = [None, "inferno", None, None, "gray", "gray", "magma", None]
for a, (img, title), cm in zip(ax.ravel(), panels, cmaps):
    if img is None:
        a.axis("off")
        continue
    a.imshow(np.asarray(img).squeeze(), cmap=cm)
    a.set_title(title, color="white", fontsize=10)
    a.axis("off")
if lock.get("bbox") is not None:
    x0, y0, x1, y1 = lock["bbox"]
    import matplotlib.patches as mpatches

    ax[0, 0].add_patch(
        mpatches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="#ff3030", lw=2)
    )
    ax[0, 0].plot(lock["cx"], lock["cy"], "+", color="#ffd000", ms=14, mew=2)
plt.tight_layout()
plt.show()

## 5 · Live AMD iGPU telemetry

The GPU monitor reads the real sysfs counters (`gpu_busy_percent`, `mem_info_vram_*`, hwmon temp) of the Radeon 8060S. These same numbers feed the “AI BRAIN” HUD panel in the next section, so you can watch the iGPU load move as the LLM plans and the kernels run.

In [ ]:
time.sleep(1.5)  # let the GPU monitor thread take a couple of samples
g = demo.GPU_STATE
print(f"GPU source : {g['source']}")
print(f"GPU busy   : {g['gpu_pct']:.0f}%")
print(
    f"VRAM       : {g['vram_used_mb']:.0f} / {g['vram_total_mb']:.0f} MB  ({g['vram_pct']:.0f}%)"
)
print(f"Temp       : {g['temp_c']:.0f} C")
if not g.get("available"):
    print("(sysfs counters not exposed here — the HUD panel will show placeholders)")

## 6 · Run the full agent — and embed the video inline

This is the payoff. We drive the **same** `parse_plan → execute_plan` loop the live demo uses — now using **your** kernels from § 4 — capturing every composited HUD frame (live scene on the left, the “AI BRAIN” plan/telemetry panel on the right), and write them to an MP4 that plays **right here in the notebook**.

`COMMANDS` runs a prerecorded list by default, including the triple stack **blue → red → green**. Edit it to whatever you want, or use § 7 to type commands live.

In [ ]:
import imageio
from IPython.display import Video, display

COMMANDS = ["stack blue on red on green", "sort the cubes"]  # <-- edit me
OUT_MP4 = "advancing_ai_agent.mp4"

# A sink queue we drain into a frame list (BGR HUD frame -> RGB for imageio).
frames_rgb = []
display_q = queue.Queue()
_stop = threading.Event()


def _drain():
    while not _stop.is_set() or not display_q.empty():
        try:
            f = display_q.get(timeout=0.1)
        except queue.Empty:
            continue
        frames_rgb.append(f[:, :, ::-1].copy())  # BGR -> RGB


_drainer = threading.Thread(target=_drain, daemon=True)
_drainer.start()

# Reset arm, then run each natural-language command through the real pipeline.
demo.go_home(franka, scene, cam, display_q, motors_dof, fingers_dof)
for cmd in COMMANDS:
    g0 = float(demo.GPU_STATE.get("gpu_pct", 0.0))
    print(f"\n[YOU] {cmd}")
    print("      LLM PLANNING ...", end="", flush=True)
    t0 = time.time()
    plan = demo.parse_plan(cmd).get("plan", [])
    print(
        f'  {((time.time()-t0)*1000):.0f} ms  | GPU {g0:.0f}%% -> {float(demo.GPU_STATE.get("gpu_pct",0)):.0f}%%'.replace(
            "%%", "%"
        )
    )
    print("      [LLM]", plan)
    if plan:
        demo.execute_plan(plan, franka, scene, cam, display_q, motors_dof, fingers_dof)

_stop.set()
_drainer.join(timeout=5)
print(f"\ncaptured {len(frames_rgb)} HUD frames")

# Write MP4 (H.264, yuv420p == browser-friendly) and embed inline.
with imageio.get_writer(
    OUT_MP4, fps=30, codec="libx264", quality=8, pixelformat="yuv420p"
) as w:
    for f in frames_rgb:
        w.append_data(f)
print("wrote", OUT_MP4)
display(Video(OUT_MP4, embed=True, width=960, html_attributes="controls autoplay loop"))

## 7 · Extras — type your own commands live (optional)

The demo above runs the prerecorded list. If you'd rather **talk to the arm interactively**, run the cell below and type commands at the `[YOU] >` prompt — they run against the *same* persistent scene, through the same LLM planner and your § 4 kernels. Try `pick the red cube`, `stack blue on red`, `sort the cubes`, `home`. Type `quit` (or press Enter on an empty line) to stop.

**When you quit, it saves everything you did to `interactive_session.mp4` and embeds it right here** — so you can play the run back and check whether the arm actually did the right thing.

> This is opt-in: it blocks on keyboard input, so leave it unrun when you just want the standard demo.

In [ ]:
import imageio
from IPython.display import Video, display


def interactive_agent(out_mp4="interactive_session.mp4"):
    """Typed prompt loop against the live scene. On quit it saves + embeds the
    full run so you can review it. 'quit'/empty line exits."""
    sink = queue.Queue()  # unbounded: push_frame never blocks
    frames_rgb = []  # every composited HUD frame, in order

    def _drain():  # pull whatever the agent has rendered so far
        while not sink.empty():
            f = sink.get_nowait()
            frames_rgb.append(f[:, :, ::-1].copy())  # BGR -> RGB

    print(
        "Interactive agent ready. Commands: pick / stack / sort / home. quit to exit."
    )
    while True:
        try:
            cmd = input("[YOU] > ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n(stopped)")
            break
        if not cmd or cmd.lower() in ("quit", "exit", "q"):
            print("bye")
            break
        g0 = float(demo.GPU_STATE.get("gpu_pct", 0.0))
        t0 = time.time()
        plan = demo.parse_plan(cmd).get("plan", [])
        print(
            f'[LLM {((time.time()-t0)*1000):.0f} ms | GPU {g0:.0f}%% -> {float(demo.GPU_STATE.get("gpu_pct",0)):.0f}%%] {plan}'.replace(
                "%%", "%"
            )
        )
        if plan:
            demo.execute_plan(plan, franka, scene, cam, sink, motors_dof, fingers_dof)
            demo.go_home(franka, scene, cam, sink, motors_dof, fingers_dof)
        _drain()  # capture this command's frames (don't discard!)
        print(f"   done.  ({len(frames_rgb)} frames captured so far)")
    # --- on quit: dump + embed the whole session so you can check it ----------
    _drain()
    if not frames_rgb:
        print("no frames captured (did you run any commands before quitting?)")
        return
    with imageio.get_writer(
        out_mp4, fps=30, codec="libx264", quality=8, pixelformat="yuv420p"
    ) as w:
        for f in frames_rgb:
            w.append_data(f)
    print(f"saved {len(frames_rgb)} frames -> {out_mp4}")
    display(
        Video(out_mp4, embed=True, width=960, html_attributes="controls autoplay loop")
    )


interactive_agent()  # <-- run this cell to start the typed prompt loop
#     (it saves + embeds interactive_session.mp4 on quit)

## 8 · Recap

You just watched, end to end:

1. **LLM** (§ 2) turn English into a JSON plan on the AMD iGPU — with the inference time and GPU busy % visible.
2. **Genesis** (§ 3) provide the physical gold Franka arm and cubes.
3. **Your HIP/ROCm kernels** (§ 4) — authored in the notebook and patched into the agent — turn raw camera data into the geometry and color locks that aim the grasp.
4. The **full agent** (§ 6) fuse all of it — plan, telemetry, tactile grip gate — into one mission-control video, then (§ 7) take your own typed commands live and save that run too.

**To go further:** rewrite a kernel in § 4 and re-run § 6 to see the arm behave differently — e.g. widen `tol` in `hip_color_lock`, or change `secure_taxels` in `hip_tactile_reduce` (how many taxels must fire before the grip is trusted).